# Autodiff from scratch

This notebook implements a small automatic differentiation engine, inspired by `micrograd`, to show how machine learning libraries compute gradients behind scalar operations.

The central idea is to build a computational graph during the forward pass. Each operation creates a new object that stores:

- the computed numerical value;
- the previous nodes that produced that value;
- a local backward function, responsible for propagating derivatives to the parent nodes.

Then, calling `backward()` on the final result applies the chain rule in reverse topological order.


In [1]:
class Value(object):
    def __init__(self, data, children=(), op=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(children)
        self._op = op

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Value(max(0, self.data), (self,), 'relu')
        def _backward():
            self.grad += (1.0 if out.data > 0 else 0.0) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        self.grad = 1.0
        for v in reversed(topo):
            v._backward()
    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)

    def __radd__(self, other):
        return self + other

    def __rmul__(self, other):
        return self * other

    def __rsub__(self, other):
        return other + (-self)

    def __pow__(self, n):
        out = Value(self.data ** n, (self,), f'**{n}')
        def _backward():
            self.grad += n * (self.data ** (n - 1)) * out.grad
        out._backward = _backward
        return out

    def __truediv__(self, other):
        return self * (other ** -1) if isinstance(other, Value) else self * (Value(other) ** -1)

    def exp(self):
        import math
        e = math.exp(self.data)
        out = Value(e, (self,), 'exp')
        def _backward():
            self.grad += e * out.grad
        out._backward = _backward
        return out

    def log(self):
        import math
        out = Value(math.log(self.data), (self,), 'log')
        def _backward():
            self.grad += (1.0 / self.data) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        import math
        t = math.tanh(self.data)
        out = Value(t, (self,), 'tanh')
        def _backward():
            self.grad += (1 - t ** 2) * out.grad
        out._backward = _backward
        return out

## Initial example: building a graph

The variables below will be used to build a simple expression. Each number is wrapped in a `Value` object, so it also carries a `grad` field that will store the derivative of the final result with respect to that value.


In [2]:
a = Value(2)
b = Value(3)
c = Value(4)
d = Value(5)

The expression being computed is:

$$
L = ((a \cdot b) + c) \cdot d
$$

While computing `e`, `f`, and `L`, the computational graph is created implicitly. Each node knows which values came before it and which operation was applied.


In [3]:
e = a * b
f = e + c
L = f * d

Before propagating gradients, we define the derivative of the output with respect to itself:

$$
\frac{\partial L}{\partial L} = 1
$$

This is the starting point of the backward pass.


In [4]:
L.grad = 1

With `L.backward()`, the notebook traverses the graph in reverse order and applies the chain rule. At the end, each input variable will have in `grad` the derivative of `L` with respect to it.


In [5]:
L.backward()

After the backward pass, you can inspect `a.grad`, `b.grad`, `c.grad`, and `d.grad` to see the effect of each variable on `L`. The larger the absolute value of the gradient, the higher the local sensitivity of the output with respect to that variable.


## Test with a small expression

Here we use a more direct expression to validate whether the `Value` class is propagating gradients correctly:

$$
L = a \cdot b + c
$$

The expected gradients are:

- $\frac{\partial L}{\partial a} = b$
- $\frac{\partial L}{\partial b} = a$
- $\frac{\partial L}{\partial c} = 1$


In [6]:
a = Value(2.0)
b = Value(3.0)
c = Value(4.0)

L = a * b + c

print(L.data)

L.backward()

print("a:", a.grad)
print("b:", b.grad)
print("c:", c.grad)

10.0
a: 3.0
b: 2.0
c: 1.0


## Minimal neural network

With the `Value` class, it is already possible to build neural network components. The implementation below defines three blocks:

- `Neuron`: computes a linear combination of the inputs and applies `tanh`;
- `Layer`: groups multiple neurons;
- `MLP`: stacks layers to form a feedforward network.

Because weights and biases are also `Value` objects, the same autodiff mechanism computes the gradients for all model parameters.


In [7]:
import random

class Neuron:
    def __init__(self, n_inputs):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(n_inputs)]
        self.b = Value(0.0)

    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.tanh()

    def parameters(self):
        return self.w + [self.b]

class Layer:
    def __init__(self, n_inputs, n_outputs):
        self.neurons = [Neuron(n_inputs) for _ in range(n_outputs)]

    def __call__(self, x):
        return [n(x) for n in self.neurons]

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

class MLP:
    def __init__(self, sizes):
        self.layers = [Layer(sizes[i], sizes[i+1]) for i in range(len(sizes)-1)]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x[0] if len(x) == 1 else x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

## Training on the XOR problem

This block trains a small MLP to approximate the XOR pattern. The loop follows the classic training cycle:

1. compute predictions in the forward pass;
2. measure the error with a squared loss;
3. reset old gradients;
4. call `loss.backward()` to compute new gradients;
5. update the parameters with gradient descent.

The targets use `-1` and `1` because the model output goes through `tanh`, whose natural range is approximately $[-1, 1]$.


In [8]:
random.seed(42)
model = MLP([2, 4, 1])  # 2 inputs, 4 hidden neurons, 1 output

xs = [[0, 0], [0, 1], [1, 0], [1, 1]]
ys = [-1, 1, 1, -1]  # XOR pattern (using -1/1 for tanh)

for step in range(100):
    preds = [model(x) for x in xs]
    loss = sum((p - y) ** 2 for p, y in zip(preds, ys))

    for p in model.parameters():
        p.grad = 0.0
    loss.backward()

    lr = 0.05
    for p in model.parameters():
        p.data -= lr * p.grad

    if step % 20 == 0:
        print(f"step {step:3d}  loss = {loss.data:.4f}")

print("\nPredictions after training:")
for x, y in zip(xs, ys):
    print(f"  input={x}  target={y:2d}  pred={model(x).data:6.3f}")

step   0  loss = 4.1491
step  20  loss = 2.9166
step  40  loss = 1.4733
step  60  loss = 0.6011
step  80  loss = 0.2936

Predictions after training:
  input=[0, 0]  target=-1  pred=-0.853
  input=[0, 1]  target= 1  pred= 0.771
  input=[1, 0]  target= 1  pred= 0.801
  input=[1, 1]  target=-1  pred=-0.753


## Numerical gradient check

To test an autodiff implementation, it is common to compare the automatically computed gradient with a finite-difference approximation:

$$
f'(x) \approx \frac{f(x+h) - f(x-h)}{2h}
$$

If the difference between both values is small, this increases confidence that the backward rules are implemented correctly.


In [8]:
def gradient_check(build_expr, x_val, h=1e-7):
    x = Value(x_val)
    y = build_expr(x)
    y.backward()
    autodiff_grad = x.grad

    y_plus = build_expr(Value(x_val + h)).data
    y_minus = build_expr(Value(x_val - h)).data
    numerical_grad = (y_plus - y_minus) / (2 * h)

    diff = abs(autodiff_grad - numerical_grad)
    return autodiff_grad, numerical_grad, diff

In [9]:
def expr(x):
    return (x ** 3 + x * 2 + 1).tanh()

ad, num, diff = gradient_check(expr, 0.5)
print(f"Autodiff:  {ad:.8f}")
print(f"Numerical: {num:.8f}")
print(f"Difference: {diff:.2e}")
# Difference should be < 1e-5

Autodiff:  0.15252426
Numerical: 0.15252426
Difference: 3.66e-10


## Testing supported operations

The next examples check specific operations implemented in `Value`: multiplication, addition, `relu`, powers, and `tanh`. These tests are useful because each operation needs its own local derivative rule.


In [10]:
x1 = Value(2.0)
x2 = Value(3.0)
a = x1 * x2          # a = 6.0
b = a + Value(1.0)    # b = 7.0
y = b.relu()          # y = 7.0

y.backward()

print(f"y = {y.data}")          # 7.0
print(f"dy/dx1 = {x1.grad}")   # 3.0 (= x2)
print(f"dy/dx2 = {x2.grad}")   # 2.0 (= x1)

y = 7.0
dy/dx1 = 3.0
dy/dx2 = 2.0


In [17]:
x1 = Value(2.0)
a = x1 ** 3           # a = 27.0

a.backward()
print(f"da/dx1 = {x1.grad}")   # 27.0 (= 3 * x1^2)

da/dx1 = 12.0


In [21]:
x1 = Value(0.0)
x2 = Value(2.0)

y1 = x1.tanh()
y2 = x2.tanh()

y1.backward()
print(f"dy1/dx1 = {x1.grad}")   # 1.
y2.backward()
print(f"dy2/dx2 = {x2.grad}")   # 0.

dy1/dx1 = 1.0
dy2/dx2 = 0.07065082485316443


## Comparison with PyTorch

PyTorch uses the same general principle: it builds a computational graph during the forward pass and propagates gradients with `backward()`. The comparison below confirms that the gradients from the manual implementation match a real library.


In [33]:
import torch

x1 = torch.tensor(2.0, requires_grad=True)
x2 = torch.tensor(3.0, requires_grad=True)
a = x1 * x2
b = a + 1.0
y = torch.relu(b)
y.backward()

print(f"PyTorch dy/dx1 = {x1.grad.item()}")  # 3.0
print(f"PyTorch dy/dx2 = {x2.grad.item()}")  # 2.0

PyTorch dy/dx1 = 3.0
PyTorch dy/dx2 = 2.0


## A simple neuron

Now we compute the gradient of a single ReLU unit:

$$
y = \text{ReLU}(w_1x_1 + w_2x_2 + b)
$$

This example resembles the basic unit of a neural network and shows how gradients flow both to inputs and to parameters.


In [70]:
x1 = Value(1.0)
x2 = Value(3.0)
w1 = Value(4)
w2 = Value(1.0)
b = Value(1.0)

In [62]:
y = (w1 * x1 + w2 * x2 + b).relu()

y.backward()

for v in [x1, x2, w1, w2, b]:
    print(f"grad: {v.grad:.4f}")

grad: 4.0000
grad: 1.0000
grad: 1.0000
grad: 3.0000
grad: 1.0000


The same computation is repeated in PyTorch to compare the results. If the printed gradients match the implementation with `Value`, the chain rule was applied correctly in this case.


In [64]:
x1 = torch.tensor(1.0, requires_grad=True)
x2 = torch.tensor(3.0, requires_grad=True)
w1 = torch.tensor(4.0, requires_grad=True)
w2 = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)

In [65]:
y = (w1 * x1 + w2 * x2 + b).relu()
y.backward()

for v in [x1, x2, w1, w2, b]:
    print(f"grad: {v.grad.item():.4f}")


grad: 4.0000
grad: 1.0000
grad: 1.0000
grad: 3.0000
grad: 1.0000


# Dual numbers and forward mode

Besides the reverse mode used in `Value`, there is another form of automatic differentiation: forward mode. It can be implemented with dual numbers.

A dual number has the form:

$$
a + b\varepsilon
$$

where `a` is the ordinary value, `b` stores derivative information, and $\varepsilon$ is an infinitesimal symbol with the special property:

$$
\varepsilon^2 = 0
$$

This property makes derivatives appear naturally when we evaluate a function. For example, if we evaluate a function at $x + 1\varepsilon$, the result has the form:

$$
f(x + \varepsilon) = f(x) + f'(x)\varepsilon
$$

So the real part gives the function value, while the coefficient of $\varepsilon$ gives the derivative. This is why dual numbers are useful for automatic differentiation: the derivative is propagated together with the value during the forward pass.

While reverse mode is efficient for many inputs and one scalar output, forward mode is simple and efficient when there are few inputs or when we want directional derivatives.


## Creating the `Dual` class

The class below stores two values at the same time:

- `value`: the normal value of the expression;
- `derivative`: the derivative of the expression with respect to a chosen variable.

Addition, multiplication, and power operations update both fields at the same time. This implements autodiff in forward mode.


In [ ]:
class Dual(object):
    def __init__(self, value, derivative=0.0):
        self.value = value
        self.derivative = derivative

    def __repr__(self):
        return f"Dual(value={self.value:.4f}, derivative={self.derivative:.4f})"
    def __add__(self, other):
        other = other if isinstance(other, Dual) else Dual(other)
        out = Dual(self.value + other.value, self.derivative + other.derivative)
        return out
    def __mul__(self, other):
        other = other if isinstance(other, Dual) else Dual(other)
        out = Dual(self.value * other.value,
                   self.derivative * other.value + self.value * other.derivative)
        return out
    def __pow__(self, n):
        out = Dual(self.value ** n, n * (self.value ** (n - 1)) * self.derivative)
        return out


A dual number with initial derivative equal to `1.0` represents the variable with respect to which we want to differentiate. For example, if `a = Dual(2.0, 1.0)`, then we are evaluating a function at $a = 2$ and assuming $da/da = 1$.


In [74]:
a = Dual(2.0, 1.0)  # value=2.0, derivative=1.0 (da/dx = 1)
a

Dual(value=2.0000, derivative=1.0000)

# Basic comparison


Let's test both approaches with the function:

$$
f(x) = x^2 + 2x + 1
$$

The analytical derivative is:

$$
f'(x) = 2x + 2
$$

At the point $x = 2$, the expected gradient is $6$.


With dual numbers, the derivative comes out together with the function value in a single pass for one input variable.


In [103]:
d = Dual(2,1)
a = d ** 2
b =  d * 2
c = 1
f = a + b + c
print(f) 

Dual(value=9.0000, derivative=6.0000)


Now we perform the same calculation with `Value`, using reverse mode. The result should match the derivative computed by the dual numbers.


In [88]:
x = Value(3)
f = x ** 2 + 2 * x + 1
f.backward()
f, x.grad

(Value(data=16.0000, grad=1.0000), 8.0)

# Example: many inputs and one output


Consider the function:

$$
f(x_1, x_2, x_3) = x_1^2 + x_2^2 + x_3^2
$$


We want to compute the full gradient:

$$
\nabla f = \left(\frac{\partial f}{\partial x_1}, \frac{\partial f}{\partial x_2}, \frac{\partial f}{\partial x_3}\right)
$$


We will evaluate this gradient at the point:

$$
(2, 3, 4)
$$


Since $\frac{d}{dx}x^2 = 2x$, we know that the expected result is:

$$
\nabla f(2, 3, 4) = (4, 6, 8)
$$


## 1. Forward mode with dual numbers


In forward mode, to obtain the full gradient of a function with three inputs, we run the function three times. In each execution, we choose one variable with initial derivative `1.0` and the others with `0.0`.


In [91]:
import numpy as np

In [108]:
def f(x: Dual, y: Dual, z: Dual):
    return x ** 2 + y**2 + z ** 2

The block below builds these three versions of the inputs. Each row of `dual_sets` corresponds to one derivative direction:

- first execution: derivative with respect to $x_1$;
- second execution: derivative with respect to $x_2$;
- third execution: derivative with respect to $x_3$.


In [92]:
xs = [2, 3, 4]

dual_sets = []

for i in range(len(xs)):
    current = []

    for j, x in enumerate(xs):
        deriv = 1.0 if i == j else 0.0
        current.append(Dual(x, deriv))

    dual_sets.append(current)

In [107]:
dual_sets[0]

[Dual(value=2.0000, derivative=1.0000),
 Dual(value=3.0000, derivative=0.0000),
 Dual(value=4.0000, derivative=0.0000)]

In [113]:
f(*dual_sets[0])

Dual(value=29.0000, derivative=4.0000)

In [114]:
f(*dual_sets[1])

Dual(value=29.0000, derivative=6.0000)

In [115]:
f(*dual_sets[2])

Dual(value=29.0000, derivative=8.0000)

## 2. Reverse mode with `Value`


With `Value`, the same function with multiple inputs needs only one backward pass to obtain all gradients of the scalar output. This is why reverse mode is widely used in neural network training: many parameters, one scalar loss.


In [116]:
x = Value(2)
y = Value(3)
z = Value(4)

In [117]:
f = x**2 + y**2 + z**2

In [118]:
f.backward()
x.grad, y.grad, z.grad

(4.0, 6.0, 8.0)